In [15]:
!pip uninstall -y numpy

In [1]:
!pip install numpy==1.23.5
!pip install gym==0.26.2
!pip install gym-super-mario-bros==7.4.0
!pip install nes-py==8.2.1

!pip install stable-baselines3==2.1.0
!pip install tensorboard
!pip install matplotlib seaborn pandas


  Using cached numpy-1.23.5.tar.gz (10.7 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
import gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import List, Optional
import json
from PIL import Image

from nes_py.wrappers import JoypadSpace
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT, COMPLEX_MOVEMENT, RIGHT_ONLY
from gym.spaces import Box

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, CheckpointCallback
from stable_baselines3.common.monitor import Monitor



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
class SkipFrame(gym.Wrapper):
    """Return only every `skip`-th frame to speed up training"""
    def __init__(self, env, skip=4):
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        done = False
        for i in range(self._skip):
            obs, reward, done, truncated, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        return obs, total_reward, done, truncated, info

class GrayScaleObservation(gym.ObservationWrapper):
    """Convert RGB observation to grayscale"""
    def __init__(self, env):
        super().__init__(env)
        obs_shape = self.observation_space.shape[:2]
        self.observation_space = Box(low=0, high=255, shape=obs_shape, dtype=np.uint8)

    def observation(self, observation):
        observation = np.dot(observation[...,:3], [0.2989, 0.5870, 0.1140])
        return observation.astype(np.uint8)

class ResizeObservation(gym.ObservationWrapper):
    """Resize observation to 84x84"""
    def __init__(self, env, shape=84):
        super().__init__(env)
        if isinstance(shape, int):
            self.shape = (shape, shape)
        else:
            self.shape = tuple(shape)
        obs_shape = self.shape + (1,) if len(env.observation_space.shape) == 2 else self.shape
        self.observation_space = Box(low=0, high=255, shape=obs_shape, dtype=np.uint8)

    def observation(self, observation):
        observation = Image.fromarray(observation)
        observation = observation.resize(self.shape, Image.BILINEAR)
        observation = np.array(observation)
        if len(observation.shape) == 2:
            observation = np.expand_dims(observation, -1)
        return observation




In [ ]:
@dataclass
class TrainingConfig:
    """Central configuration for all training parameters"""

    # Environment settings
    level: str = 'SuperMarioBros-1-1-v0'
    action_space_name: str = 'SIMPLE_MOVEMENT'

    # Preprocessing
    skip_frames: int = 4
    grayscale: bool = True
    resize: int = 84
    frame_stack: int = 4

    # PPO Hyperparameters
    learning_rate: float = 0.0003
    n_steps: int = 512
    batch_size: int = 64
    n_epochs: int = 10
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_range: float = 0.2
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5

    # Training duration
    total_timesteps: int = 1_000_000

    # Logging and saving
    log_interval: int = 1
    save_freq: int = 50_000
    tensorboard_log: str = "./logs/"
    model_save_path: str = "./models/"
    experiment_name: str = "baseline_ppo"

    # Evaluation
    eval_freq: int = 25_000
    n_eval_episodes: int = 5

    def to_dict(self):
        return asdict(self)

    def save(self, filepath):
        with open(filepath, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def load(cls, filepath):
        with open(filepath, 'r') as f:
            return cls(**json.load(f))

In [ ]:
class TrainingMetricsCallback(BaseCallback):
    """Custom callback for logging additional metrics during training"""
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []
        self.episode_x_positions = []

    def _on_step(self) -> bool:
        if self.locals.get('dones')[0]:
            info = self.locals['infos'][0]
            self.episode_rewards.append(info.get('episode', {}).get('r', 0))
            self.episode_lengths.append(info.get('episode', {}).get('l', 0))
            self.episode_x_positions.append(info.get('x_pos', 0))

            if len(self.episode_rewards) > 0:
                self.logger.record("rollout/ep_rew_mean", np.mean(self.episode_rewards[-100:]))
                self.logger.record("rollout/ep_len_mean", np.mean(self.episode_lengths[-100:]))
                self.logger.record("rollout/max_x_pos", np.max(self.episode_x_positions[-100:]))

        return True

class ProgressCallback(BaseCallback):
    """Callback to display training progress"""
    def __init__(self, total_timesteps, verbose=0):
        super().__init__(verbose)
        self.total_timesteps = total_timesteps

    def _on_step(self) -> bool:
        if self.n_calls % 10000 == 0:
            progress = (self.num_timesteps / self.total_timesteps) * 100
            print(f"Progress: {progress:.1f}% ({self.num_timesteps}/{self.total_timesteps} timesteps)")
        return True


In [ ]:
def create_mario_env(level='SuperMarioBros-1-1-v0', action_space=SIMPLE_MOVEMENT,
                     skip_frames=4, grayscale=True, resize=84, frame_stack=4):
    """
    Create a fully wrapped Mario environment

    Args:
        level: The Mario level to play
        action_space: Which action space to use
        skip_frames: How many frames to skip
        grayscale: Whether to convert to grayscale
        resize: Target size for observation
        frame_stack: How many frames to stack
    """
    env = gym.make(level, apply_api_compatibility=True)
    env = JoypadSpace(env, action_space)

    if skip_frames > 1:
        env = SkipFrame(env, skip=skip_frames)

    if grayscale:
        env = GrayScaleObservation(env)

    if resize:
        env = ResizeObservation(env, shape=resize)

    env = DummyVecEnv([lambda: env])

    if frame_stack > 1:
        env = VecFrameStack(env, n_stack=frame_stack, channels_order='last')

    return env

In [ ]:
def get_action_space(name):
    """Get action space by name"""
    spaces = {
        'RIGHT_ONLY': RIGHT_ONLY,
        'SIMPLE_MOVEMENT': SIMPLE_MOVEMENT,
        'COMPLEX_MOVEMENT': COMPLEX_MOVEMENT
    }
    return spaces[name]


In [ ]:
def train_baseline_agent(config: TrainingConfig):
    """Train a baseline PPO agent with the given configuration"""

    # Create directories
    os.makedirs(config.model_save_path, exist_ok=True)
    os.makedirs(config.tensorboard_log, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Training: {config.experiment_name}")
    print(f"{'='*60}\n")

    # Create environment
    print("Creating environment...")
    action_space = get_action_space(config.action_space_name)

    env = create_mario_env(
        level=config.level,
        action_space=action_space,
        skip_frames=config.skip_frames,
        grayscale=config.grayscale,
        resize=config.resize,
        frame_stack=config.frame_stack
    )

    print(f"✓ Environment created: {config.level}")
    print(f"  - Action space: {config.action_space_name} ({env.action_space.n} actions)")
    print(f"  - Observation shape: {env.observation_space.shape}")

    # Create PPO model
    print("\nCreating PPO model...")
    model = PPO(
        "CnnPolicy",
        env,
        learning_rate=config.learning_rate,
        n_steps=config.n_steps,
        batch_size=config.batch_size,
        n_epochs=config.n_epochs,
        gamma=config.gamma,
        gae_lambda=config.gae_lambda,
        clip_range=config.clip_range,
        ent_coef=config.ent_coef,
        vf_coef=config.vf_coef,
        max_grad_norm=config.max_grad_norm,
        verbose=1,
        tensorboard_log=config.tensorboard_log,
    )

    print("✓ PPO model created")
    print(f"  - Total parameters: {sum(p.numel() for p in model.policy.parameters()):,}")

    # Setup callbacks
    checkpoint_callback = CheckpointCallback(
        save_freq=config.save_freq,
        save_path=config.model_save_path,
        name_prefix=config.experiment_name
    )

    metrics_callback = TrainingMetricsCallback()
    progress_callback = ProgressCallback(config.total_timesteps)

    callback_list = CallbackList([checkpoint_callback, metrics_callback, progress_callback])

    # Save configuration
    config.save(os.path.join(config.model_save_path, f"{config.experiment_name}_config.json"))

    # Train
    print(f"\nStarting training for {config.total_timesteps:,} timesteps...")
    print(f"Tensorboard logs: {config.tensorboard_log}")
    print(f"Model checkpoints: {config.model_save_path}\n")

    start_time = datetime.now()

    model.learn(
        total_timesteps=config.total_timesteps,
        callback=callback_list,
        tb_log_name=config.experiment_name
    )

    end_time = datetime.now()
    training_time = end_time - start_time

    # Save final model
    final_model_path = os.path.join(config.model_save_path, f"{config.experiment_name}_final")
    model.save(final_model_path)

    print(f"\n{'='*60}")
    print(f"Training Complete!")
    print(f"{'='*60}")
    print(f"Total training time: {training_time}")
    print(f"Final model saved to: {final_model_path}")

    env.close()

    return model, metrics_callback



In [ ]:
def evaluate_agent(model, config, n_episodes=10, render=False):
    """Evaluate a trained agent"""
    print(f"\nEvaluating agent for {n_episodes} episodes...")

    action_space = get_action_space(config.action_space_name)
    eval_env = create_mario_env(
        level=config.level,
        action_space=action_space,
        skip_frames=config.skip_frames,
        grayscale=config.grayscale,
        resize=config.resize,
        frame_stack=config.frame_stack
    )

    episode_rewards = []
    episode_lengths = []
    episode_x_positions = []
    level_completions = []

    for episode in range(n_episodes):
        obs = eval_env.reset()
        done = False
        episode_reward = 0
        episode_length = 0
        max_x = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = eval_env.step(action)
            episode_reward += reward[0]
            episode_length += 1
            max_x = max(max_x, info[0].get('x_pos', 0))

        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
        episode_x_positions.append(max_x)
        level_completions.append(info[0].get('flag_get', False))

        print(f"Episode {episode+1}: Reward={episode_reward:.0f}, "
              f"Length={episode_length}, MaxX={max_x}, "
              f"Completed={'Yes' if level_completions[-1] else 'No'}")

    eval_env.close()

    print(f"\n{'='*60}")
    print("Evaluation Summary:")
    print(f"{'='*60}")
    print(f"Average Reward: {np.mean(episode_rewards):.2f} ± {np.std(episode_rewards):.2f}")
    print(f"Average Length: {np.mean(episode_lengths):.2f} ± {np.std(episode_lengths):.2f}")
    print(f"Average Max X: {np.mean(episode_x_positions):.2f} ± {np.std(episode_x_positions):.2f}")
    print(f"Completion Rate: {sum(level_completions)/len(level_completions)*100:.1f}%")

    return {
        'rewards': episode_rewards,
        'lengths': episode_lengths,
        'x_positions': episode_x_positions,
        'completions': level_completions
    }


In [ ]:
def plot_training_metrics(metrics_callback, save_path=None):
    """Plot training metrics from callback"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Episode rewards
    axes[0, 0].plot(metrics_callback.episode_rewards)
    axes[0, 0].set_title('Episode Rewards Over Time')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Total Reward')
    axes[0, 0].grid(True, alpha=0.3)

    # Moving average of rewards
    window = 100
    if len(metrics_callback.episode_rewards) >= window:
        moving_avg = pd.Series(metrics_callback.episode_rewards).rolling(window=window).mean()
        axes[0, 1].plot(moving_avg)
        axes[0, 1].set_title(f'Episode Rewards ({window}-Episode Moving Average)')
        axes[0, 1].set_xlabel('Episode')
        axes[0, 1].set_ylabel('Avg Reward')
        axes[0, 1].grid(True, alpha=0.3)

    # Episode lengths
    axes[1, 0].plot(metrics_callback.episode_lengths)
    axes[1, 0].set_title('Episode Lengths Over Time')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('Length (steps)')
    axes[1, 0].grid(True, alpha=0.3)

    # Max X positions
    axes[1, 1].plot(metrics_callback.episode_x_positions)
    axes[1, 1].set_title('Max X Position Reached Per Episode')
    axes[1, 1].set_xlabel('Episode')
    axes[1, 1].set_ylabel('X Position')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")

    plt.show()

def plot_evaluation_results(eval_results, save_path=None):
    """Plot evaluation results"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Rewards distribution
    axes[0].hist(eval_results['rewards'], bins=10, edgecolor='black')
    axes[0].axvline(np.mean(eval_results['rewards']), color='red',
                    linestyle='--', label=f"Mean: {np.mean(eval_results['rewards']):.1f}")
    axes[0].set_title('Episode Rewards Distribution')
    axes[0].set_xlabel('Reward')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # X positions
    axes[1].bar(range(len(eval_results['x_positions'])), eval_results['x_positions'])
    axes[1].set_title('Max X Position Per Episode')
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('X Position')
    axes[1].grid(True, alpha=0.3)

    # Completion rate
    completions = eval_results['completions']
    labels = ['Completed', 'Failed']
    sizes = [sum(completions), len(completions) - sum(completions)]
    axes[2].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    axes[2].set_title('Level Completion Rate')

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")

    plt.show()

In [2]:
print("Testing environment creation and stepping...")

# Create test environment
test_env = create_mario_env(
    level='SuperMarioBros-1-1-v0',
    action_space=SIMPLE_MOVEMENT,
    skip_frames=4,
    grayscale=True,
    resize=84,
    frame_stack=4
)

# Reset and step
state = test_env.reset()
print(f"✓ Environment created successfully!")
print(f"  - State shape: {state.shape}")
print(f"  - Action space size: {test_env.action_space.n}")

# Test a few steps
for i in range(5):
    action = test_env.action_space.sample()
    state, reward, done, info = test_env.step([action])

print("✓ Environment test passed!")
test_env.close()

Testing environment creation and stepping...


NameError: name 'create_mario_env' is not defined